# 01 — Complexity Analysis

## Why This Matters for DSA
Every choice you make in this course — which container, which algorithm, which recursion strategy — is a complexity trade-off. "It works on my test case" is not the bar; "it works within the time limit at the input size the problem promises" is. Complexity analysis is the tool that lets you predict whether code will finish in milliseconds or hours, BEFORE you run it on the full-size input. Get this wrong and you'll either over-engineer trivial problems or submit a solution that times out on anything past n=10,000.

## Prerequisites
- Comfort with loops, recursion, and basic C++ syntax
- `03_STL_Deep_Dive` (helpful but not required — this notebook references `vector`'s amortized cost, which that notebook covers in more STL-specific depth)

## Learning Objectives
By the end of this notebook, you will be able to:
- State what Big-O notation actually means, and why it describes a worst-case UPPER BOUND, not an exact runtime
- Recognize the standard complexity classes (O(1), O(log n), O(n), O(n log n), O(n²), O(2ⁿ), O(n!)) from code shape alone
- Apply the two core rules for analyzing loops: add for sequential, multiply for nested
- Distinguish time complexity from space complexity, including the hidden cost of recursion's call stack
- Explain amortized analysis well enough to derive WHY `vector::push_back` is O(1) amortized, not just recite that it is
- Tell apart best case, average case, and worst case, and know which one Big-O conventionally describes
- Empirically benchmark code with `<chrono>` to verify a complexity claim when you're not sure of it on paper

## Checklist Before Moving On
- [ ] I can look at a nested loop and state its complexity without running it
- [ ] I can explain why a loop that does `i *= 2` is O(log n), not O(n)
- [ ] I know recursion has a space cost even when it doesn't allocate anything explicitly
- [ ] I can explain the aggregate-method argument for amortized O(1) `push_back` using real numbers, not just the phrase "it doubles"
- [ ] I know Big-O by convention describes worst case unless stated otherwise
- [ ] I can write a `<chrono>` benchmark that isn't silently optimized away by the compiler

## Table of Contents
1. What Big-O Actually Means
2. O(1) and O(n) — Constant and Linear Time
3. O(log n) and O(n log n) — Logarithmic and Linearithmic Time
4. O(n²), O(2ⁿ), and O(n!) — Quadratic, Exponential, and Factorial Time
5. Analyzing Loops — The Rules for Reading Code
6. Space Complexity — Auxiliary Space and the Recursion Stack
7. Amortized Analysis — The Aggregate Method
8. Best Case, Average Case, Worst Case
9. Empirical Benchmarking with `<chrono>`
10. Phase Checkpoint, Cheat Sheet, and Answer Key


## 1. What Big-O Actually Means

### The Why
"Big-O" gets thrown around loosely enough that people forget it has a precise meaning. Knowing that meaning is what lets you correctly say "this is O(n²)" instead of guessing from vibes, and it's what lets you spot when someone (including past-you) has mis-labeled something.

### Core Mechanism
- Big-O describes an **upper bound** on how an algorithm's cost grows as input size (conventionally called `n`) grows — specifically, for LARGE n. It deliberately ignores constant factors and lower-order terms, because those stop mattering once n is big enough.
- Formally: `f(n) = O(g(n))` means there exist constants `c > 0` and `n₀` such that `f(n) ≤ c · g(n)` for all `n ≥ n₀`. In plain terms: past some point, `f` never grows faster than a constant multiple of `g`.
- This is WHY `O(2n)` and `O(n)` are considered the same — the constant `2` doesn't change which growth *class* the function belongs to, and Big-O only cares about growth class.
- By convention (unless a problem says otherwise), "the complexity of this algorithm" means its **worst-case** complexity — the input that makes it work the hardest. Section 8 covers best/average/worst in more depth.
- There are two related but less commonly used notations: **Big-Ω (Omega)** is a LOWER bound (best case), and **Big-Θ (Theta)** means the upper and lower bounds match — the algorithm's growth is EXACTLY that rate, both from above and below. Most interview conversations use Big-O loosely to mean "Big-Θ, worst case" — technically imprecise, universally understood.

### Common Pitfalls
- Treating Big-O as an exact runtime formula instead of a growth-rate classification. "O(n)" doesn't tell you HOW LONG something takes — it tells you how the time changes as n changes.
- Forgetting Big-O usually means worst case by convention, then being surprised when "O(n) worst case" code is fast on typical inputs (average case can be much better than worst case).


## 2. O(1) and O(n) — Constant and Linear Time

### The Why
These are the two you'll recognize fastest, and they anchor your intuition for everything else — every other complexity class is described relative to how much WORSE (or better) it is than linear.

### Core Mechanism
- **O(1) — constant time:** the cost is the same no matter how big the input is. A single array access (`v[0]`) does exactly one step whether `v` has 10 elements or 10 million.
- **O(n) — linear time:** cost grows directly proportional to input size. Doubling the input roughly doubles the work. A single pass over every element (linear search, summing an array) is the canonical example.

### Common Pitfalls
- Assuming any single loop is automatically O(n) — it's only O(n) if the loop runs a number of times proportional to n. A loop that runs a FIXED number of times (say, always 100 iterations regardless of input) is still O(1).


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
using namespace std;

// O(1): constant time -- cost doesn't depend on input size at all
int getFirst(const vector<int>& v) {
    return v[0];   // one array access, always exactly one step regardless of v.size()
}

// O(n): linear time -- cost grows directly proportional to input size
int linearSearch(const vector<int>& v, int target) {
    for (int i = 0; i < (int)v.size(); i++) {   // worst case: check every element once
        if (v[i] == target) return i;
    }
    return -1;
}

int main() {
    vector<int> small(10, 5);
    vector<int> large(10000000, 5);

    cout << "getFirst(small)=" << getFirst(small) << "\n";
    cout << "getFirst(large)=" << getFirst(large) << "\n";
    // both calls above do the SAME amount of work -- that's what O(1) means

    cout << "linearSearch found at: " << linearSearch(small, 5) << "\n";
    // linearSearch on `large` does up to 1,000,000x more work than on `small`
    // (10,000,000 elements vs 10) -- that's what O(n) means: work scales with n

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 3. O(log n) and O(n log n) — Logarithmic and Linearithmic Time

### The Why
O(log n) is the complexity class that separates "fast" from "instant" at scale — binary search on a billion elements still finishes in about 30 steps. O(n log n) is the complexity of every efficient comparison-based sort, and recognizing it tells you sorting-based approaches are usually "fast enough" without needing to prove it from scratch each time.

### Core Mechanism
- **O(log n) — logarithmic time:** each step eliminates a CONSTANT FRACTION (usually half) of the remaining problem. Binary search is the textbook example: comparing the middle element throws away half the remaining range every time.
- The defining empirical signature: multiplying `n` by some factor `k` only ADDS a constant number of extra steps — it does NOT multiply the steps by `k`. You'll see this directly in the benchmark below: `n` grows by 64× total (16 → 1,048,576) but the step count barely grows (5 → 21).
- **O(n log n) — linearithmic time:** the complexity of efficient comparison-based sorting (`std::sort`, merge sort, heap sort). Intuition: you do roughly `n` "layers" of work, and each layer costs about `O(log n)` — for merge sort specifically, `log n` merge levels, each doing `O(n)` total merging work across that level.

### Common Pitfalls
- Assuming halving-based logic always means O(log n) even when it's wrapped in a loop over all elements — check whether EACH step also touches O(n) work; if so, you likely have O(n log n), not O(log n).


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
#include <algorithm>
using namespace std;

// O(log n): logarithmic time -- each step eliminates HALF the remaining problem
int binarySearch(const vector<int>& v, int target) {
    int lo = 0, hi = (int)v.size() - 1;
    int steps = 0;
    while (lo <= hi) {
        steps++;
        int mid = lo + (hi - lo) / 2;
        if (v[mid] == target) {
            cout << "found in " << steps << " steps (n=" << v.size() << ")\n";
            return mid;
        } else if (v[mid] < target) {
            lo = mid + 1;
        } else {
            hi = mid - 1;
        }
    }
    cout << "not found, " << steps << " steps (n=" << v.size() << ")\n";
    return -1;
}

int main() {
    // watch how steps grow: doubling n adds only ONE more step, not double the steps
    for (int n : {16, 256, 4096, 65536, 1048576}) {
        vector<int> v(n);
        for (int i = 0; i < n; i++) v[i] = i;
        binarySearch(v, n - 1);   // search for the last element (worst case)
    }

    // O(n log n): sort's complexity -- n "layers" of work, each layer doing O(log n)
    // comparisons per element on average (this is why sort beats O(n^2) approaches
    // at scale, and why almost every efficient sort you'll meet is n log n)
    vector<int> data{5, 3, 8, 1, 9, 2, 7, 4, 6};
    sort(data.begin(), data.end());   // O(n log n) -- introsort under the hood
    for (int x : data) cout << x << " ";
    cout << "\n";

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 4. O(n²), O(2ⁿ), and O(n!) — Quadratic, Exponential, and Factorial Time

### The Why
These are the complexity classes that make previously-fine code fall over as input grows — an O(n²) algorithm that's instant at n=100 can take minutes at n=100,000. Recognizing them on sight is what tells you "this approach won't survive the problem's constraints" before you've wasted time implementing it.

### Core Mechanism
- **O(n²) — quadratic time:** the signature of a nested loop where BOTH loops scale with `n` — comparing every pair of elements, bubble sort, a naive "for each element, scan the rest" pattern. Doubling `n` roughly QUADRUPLES the work (2² = 4).
- **O(2ⁿ) — exponential time:** typically comes from unmemoized recursion that branches into 2 (or more) sub-calls at every level, re-exploring overlapping work repeatedly. Naive recursive Fibonacci is the canonical example — `fib(n)` calls `fib(n-1)` and `fib(n-2)`, and those calls overlap massively without memoization. Each `+1` to `n` roughly DOUBLES the total work.
- **O(n!) — factorial time:** generating all permutations of `n` items. This grows faster than exponential — `10!` is already 3.6 million, `15!` is over a trillion. You'll only see this complexity class when a problem genuinely requires exploring every ordering (or you've accidentally written something that does).

### Common Pitfalls
- Writing naive recursive Fibonacci (or similar overlapping-subproblem recursion) without memoization and being surprised it "hangs" for inputs barely bigger than the ones that worked instantly — this IS the exponential blowup, not a bug.
- Confusing O(2ⁿ) and O(n!) — both are "very slow," but factorial grows even faster; for reference, `2²⁰ ≈ 1` million while `20! ≈ 2.4 × 10¹⁸`.


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
using namespace std;

// O(n^2): quadratic time -- a nested loop, each level scanning roughly n items
long long countPairs(const vector<int>& v) {
    long long comparisons = 0;
    for (int i = 0; i < (int)v.size(); i++) {
        for (int j = i + 1; j < (int)v.size(); j++) {
            comparisons++;   // every pair gets compared once -- n*(n-1)/2 total
        }
    }
    return comparisons;
}

// O(2^n): exponential time -- naive recursive Fibonacci, no memoization
// each call spawns 2 more calls, and the recursion tree has depth n
long long fibCalls = 0;
long long fibNaive(int n) {
    fibCalls++;
    if (n <= 1) return n;
    return fibNaive(n - 1) + fibNaive(n - 2);
}

// O(n!): factorial time -- generating all permutations of n elements
long long permCount = 0;
void permute(vector<int>& v, int k) {
    if (k == (int)v.size()) {
        permCount++;
        return;
    }
    for (int i = k; i < (int)v.size(); i++) {
        swap(v[k], v[i]);
        permute(v, k + 1);
        swap(v[k], v[i]);   // backtrack
    }
}

int main() {
    for (int n : {10, 20, 40}) {
        vector<int> v(n);
        cout << "n=" << n << " pair comparisons=" << countPairs(v) << "\n";
    }
    // doubling n roughly QUADRUPLES the comparisons -- that's the signature of O(n^2)

    for (int n : {10, 20, 30}) {
        fibCalls = 0;
        fibNaive(n);
        cout << "fib(" << n << ") took " << fibCalls << " calls\n";
    }
    // each +10 to n multiplies the call count by roughly 100x -- explosive growth,
    // this is why naive recursive fibonacci is unusable past n~40 without memoization

    for (int n : {4, 6, 8}) {
        permCount = 0;
        vector<int> v(n);
        for (int i = 0; i < n; i++) v[i] = i;
        permute(v, 0);
        cout << "n=" << n << " permutations=" << permCount << "\n";
    }
    // 8! = 40320, and it only gets worse -- 12! is already ~479 million

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 5. Analyzing Loops — The Rules for Reading Code

### The Why
This is the actual mechanical skill: given a chunk of code, assign it a complexity WITHOUT running it. Two rules cover the overwhelming majority of cases you'll meet.

### Core Mechanism
- **Rule 1 — Sequential loops: ADD, then keep the dominant term.** Two separate loops that each run over the input, one after another, cost `O(n) + O(n²) = O(n²)` overall — the largest term dominates for big `n`, so smaller terms are simply dropped when stating the final complexity.
- **Rule 2 — Nested loops: MULTIPLY.** A loop of `n` iterations, each containing another loop of `n` iterations, costs `O(n) × O(n) = O(n²)`.
- **Rule 3 — Multiplicative step loops are O(log n), not O(n).** A loop where the counter is multiplied or divided each iteration (`i *= 2`, or `i /= 2`) runs roughly `log n` times, not `n` times — this is easy to miss if you pattern-match "it's a loop" to "it's O(n)" without checking HOW the loop variable changes.
- **Rule 4 — A shrinking inner bound is still O(n²), via the triangular-number identity.** A nested loop where the inner loop's range depends on the outer index (common in pair-counting and DP table fills) sums to `n + (n-1) + ... + 1 = n(n+1)/2`. That's still quadratic — the `1/2` is a constant factor, and Big-O drops constant factors.

### Common Pitfalls
- Seeing a shrinking inner loop bound and assuming it's "less than O(n²)" because it's not always running the full `n` iterations — the triangular sum still simplifies to O(n²), just with a smaller constant.


In [ ]:
%%writefile temp.cpp
#include <iostream>
using namespace std;

int main() {
    int n = 8;

    // RULE 1: sequential loops -- ADD their complexities, then keep only the dominant term
    long long work = 0;
    for (int i = 0; i < n; i++) work++;        // O(n)
    for (int i = 0; i < n; i++)
        for (int j = 0; j < n; j++) work++;    // O(n^2)
    // total: O(n) + O(n^2) = O(n^2) -- the n^2 term dominates for large n, so the O(n) part
    // is simply dropped when stating the overall complexity
    cout << "sequential loops work=" << work << " (n=" << n << ")\n";

    // RULE 2: nested loops -- MULTIPLY their complexities
    long long nested = 0;
    for (int i = 0; i < n; i++) {          // O(n)
        for (int j = 0; j < n; j++) {      // O(n) per outer iteration
            nested++;                       // total: O(n) * O(n) = O(n^2)
        }
    }
    cout << "nested loop work=" << nested << " = n^2 = " << n*n << "\n";

    // RULE 3: a loop whose bound SHRINKS/GROWS multiplicatively each step is O(log n),
    // not O(n) -- this is easy to miss if you only look at "it's a loop" and assume O(n)
    long long logSteps = 0;
    for (int i = 1; i < n; i *= 2) {   // i doubles each time: 1, 2, 4, 8, ...
        logSteps++;
    }
    cout << "multiplicative loop steps=" << logSteps << " (log2(" << n << ")=~3)\n";

    // RULE 4: a nested loop where the inner bound depends on the outer index
    // (like counting pairs, or many DP table fills) is STILL O(n^2) -- the triangular
    // sum n + (n-1) + (n-2) + ... + 1 simplifies to n(n+1)/2, which is O(n^2), not O(n)
    long long triangular = 0;
    for (int i = 0; i < n; i++) {
        for (int j = i; j < n; j++) {
            triangular++;
        }
    }
    cout << "triangular loop work=" << triangular << " (formula n(n+1)/2=" << n*(n+1)/2 << ")\n";

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 6. Space Complexity — Auxiliary Space and the Recursion Stack

### The Why
Time complexity gets all the attention, but space complexity is just as often the actual constraint — and it has a hidden source that trips people up constantly: recursion has a real memory cost even when it never explicitly allocates anything.

### Core Mechanism
- **Auxiliary space** is extra memory used BEYOND the input itself — the input array doesn't count against your algorithm's space complexity, but any additional structures you build do.
- An iterative loop that just tracks a running total uses O(1) auxiliary space — a fixed handful of variables, regardless of input size.
- A recursive function that makes `n` nested calls before any of them returns has `n` **stack frames** alive simultaneously — that's O(n) auxiliary space, even though the function body itself never writes `new` or creates a container. Each frame holds the function's local variables and its return address, and those frames pile up until the base case is reached and unwinding begins.
- This is the single most important space-complexity fact for this course: **an iterative and a recursive solution to the same problem can have identical TIME complexity (both O(n)) while differing sharply in SPACE complexity (O(1) vs O(n))**, purely because of the call stack. "Recursive" does not mean "slower" — it usually means "more memory," specifically stack memory.

### Common Pitfalls
- Only counting explicitly-allocated data structures (`vector`, `map`, etc.) as space usage and forgetting to count recursion depth — a deep recursive call chain IS space usage, and can even cause a stack overflow on inputs an iterative version would handle fine.


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
using namespace std;

// O(1) auxiliary space: uses a fixed number of extra variables, regardless of input size
int sumIterative(const vector<int>& v) {
    int total = 0;                       // one extra variable, no matter how big v is
    for (int x : v) total += x;
    return total;
}

// O(n) auxiliary space: the recursion stack itself takes space -- n nested calls
// means n stack frames alive at once before any of them return
int sumRecursive(const vector<int>& v, int idx) {
    if (idx == (int)v.size()) return 0;
    return v[idx] + sumRecursive(v, idx + 1);   // this call must wait for the next one,
                                                  // so all n frames stack up before unwinding
}

// O(n) auxiliary space from an explicit data structure, not recursion
vector<int> doubledCopy(const vector<int>& v) {
    vector<int> result;
    result.reserve(v.size());
    for (int x : v) result.push_back(x * 2);   // allocates a whole new array proportional to n
    return result;
}

int main() {
    vector<int> v{1, 2, 3, 4, 5};

    cout << "iterative sum=" << sumIterative(v) << " -- O(1) extra space\n";
    cout << "recursive sum=" << sumRecursive(v, 0) << " -- O(n) extra space (call stack depth n)\n";

    vector<int> doubled = doubledCopy(v);
    cout << "doubled copy: ";
    for (int x : doubled) cout << x << " ";
    cout << "-- O(n) extra space (new array)\n";

    // the key distinction interviewers care about: TIME complexity of sumIterative and
    // sumRecursive is the same (O(n), both touch every element once) -- but their SPACE
    // complexity differs (O(1) vs O(n)), purely because of recursion's stack frames.
    // "recursive" does not automatically mean "worse time" -- it usually means "worse space"

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 7. Amortized Analysis — The Aggregate Method

### The Why
"`vector::push_back` is O(1) amortized" is a phrase everyone repeats, but far fewer people can actually derive WHY. Being able to derive it — with real numbers — is what separates reciting a fact from understanding it, and the same reasoning pattern (aggregate method) applies to other amortized-cost structures you'll meet later.

### Core Mechanism
- **The setup:** a dynamic array starts empty and doubles its capacity every time it's full and needs to grow. Most `push_back` calls are cheap (O(1) — just write to the next free slot). Occasionally, a `push_back` triggers a resize, which copies EVERY existing element into a new, bigger block — that single call is O(current size), i.e. expensive.
- **The aggregate method:** instead of analyzing one `push_back` in isolation, sum the TOTAL cost across `n` pushes, then divide by `n` to get the average (amortized) cost per push.
- **The math:** resizes happen when the array reaches sizes 1, 2, 4, 8, 16, ... Each resize at size `k` copies `k` elements. Total copying work across all resizes up to size `n` is `1 + 2 + 4 + ... + n/2`, a geometric series that sums to just under `n` (specifically, `< 2n`). Add the `n` cheap pushes themselves (each O(1)), and total work for `n` pushes is `O(n) + O(n) = O(n)`.
- **The conclusion:** `O(n)` total work spread across `n` pushes gives `O(n)/n = O(1)` average cost per push — this is exactly what "amortized O(1)" means. Any INDIVIDUAL push that triggers a resize is genuinely O(n) at that moment, but that expensive call is rare enough, and the cheap calls frequent enough, that the AVERAGE stays constant.

### Common Pitfalls
- Interpreting "amortized O(1)" as "every single push_back is O(1)" — it isn't; the guarantee is about the AVERAGE over many calls, not any one call in isolation. If a single call's worst-case timing matters (hard real-time systems), amortized analysis is the wrong tool and you need true worst-case bounds instead.


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
using namespace std;

// Manual re-implementation of vector's doubling strategy, WITH a cost counter,
// so we can see the aggregate-method argument for amortized O(1) push_back with real numbers
struct CountingArray {
    vector<int> data;
    long long totalCopyOps = 0;   // every element copy during a resize counts as 1 op

    void push_back(int val) {
        if (data.size() == data.capacity()) {
            size_t newCap = data.capacity() == 0 ? 1 : data.capacity() * 2;
            vector<int> newData;
            newData.reserve(newCap);
            for (int x : data) {
                newData.push_back(x);
                totalCopyOps++;      // this is the "expensive" part of a resize
            }
            data = move(newData);
        }
        data.push_back(val);
    }
};

int main() {
    // Aggregate method: sum up the TOTAL cost of n pushes, then divide by n.
    // Resizes happen at sizes 1, 2, 4, 8, 16, ... -- each resize of size k copies k elements.
    // Total copies for n pushes = 1+2+4+8+...+n/2 < n (a geometric series sums to just under n).
    // So total work for n pushes is O(n) + O(n) [the pushes themselves] = O(n).
    // Divided across n pushes: O(n)/n = O(1) per push, ON AVERAGE. That's amortized O(1).

    for (int n : {16, 256, 4096, 65536}) {
        CountingArray arr;
        for (int i = 0; i < n; i++) arr.push_back(i);
        double avgCopiesPerPush = (double)arr.totalCopyOps / n;
        cout << "n=" << n << " total copy-ops=" << arr.totalCopyOps
             << " avg copies/push=" << avgCopiesPerPush << "\n";
    }
    // notice avg copies/push stays small and roughly CONSTANT as n grows by 4096x --
    // that's the empirical signature of amortized O(1), even though any INDIVIDUAL
    // push that triggers a resize is doing real O(n) work at that moment

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 8. Best Case, Average Case, Worst Case

### The Why
The same algorithm can have wildly different costs depending on the input it's given — and knowing which case a stated complexity refers to is essential for correctly comparing two algorithms or predicting real-world behavior.

### Core Mechanism
- **Best case:** the input that makes the algorithm do the LEAST work. For linear search, that's the target being the very first element checked — O(1).
- **Worst case:** the input that makes the algorithm do the MOST work. For linear search, that's the target being last, or entirely absent — O(n). **This is what "Big-O" conventionally refers to unless a problem explicitly asks for something else.**
- **Average case:** the expected cost across a realistic distribution of inputs. For linear search over a target equally likely to be anywhere (or absent), the expected number of comparisons is about `n/2` — still O(n) after dropping the constant, but arrived at through different reasoning than the worst-case argument.
- Why this matters practically: quicksort is a famous example where worst case is O(n²) (already-sorted input with a poor pivot choice) but average case is O(n log n) — and in practice, with randomized pivot selection, you almost always see the average-case behavior. Knowing both numbers, and which one is more likely to matter for your actual input, is part of choosing the right algorithm.

### Common Pitfalls
- Quoting only a worst-case bound when discussing an algorithm's PRACTICAL behavior, without mentioning that the worst case is rare or preventable (e.g. quicksort with randomized pivots essentially never hits true O(n²) in practice) — technically correct, practically misleading.


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
using namespace std;

// linear search shows all three cases clearly depending on WHERE the target is
int linearSearch(const vector<int>& v, int target, long long &comparisons) {
    comparisons = 0;
    for (int i = 0; i < (int)v.size(); i++) {
        comparisons++;
        if (v[i] == target) return i;
    }
    return -1;
}

int main() {
    vector<int> v(1000);
    for (int i = 0; i < 1000; i++) v[i] = i;

    long long comparisons;

    // BEST CASE: target is the very first element -- O(1)
    linearSearch(v, 0, comparisons);
    cout << "best case (target at index 0): " << comparisons << " comparisons\n";

    // WORST CASE: target is the last element, or absent entirely -- O(n)
    linearSearch(v, 999, comparisons);
    cout << "worst case (target at last index): " << comparisons << " comparisons\n";
    linearSearch(v, -1, comparisons);
    cout << "worst case (target absent): " << comparisons << " comparisons\n";

    // AVERAGE CASE: target is equally likely to be anywhere -- expected ~n/2 comparisons.
    // Big-O notation, by convention, almost always describes the WORST case unless stated
    // otherwise -- "linear search is O(n)" means worst case O(n), even though its average
    // case is technically n/2 (which is still O(n) after dropping constants, but the
    // REASONING for why is different from the worst-case reasoning).
    long long totalComparisons = 0;
    for (int target = 0; target < 1000; target++) {
        linearSearch(v, target, comparisons);
        totalComparisons += comparisons;
    }
    cout << "average case (averaged over all positions): "
         << (double)totalComparisons / 1000 << " comparisons (~n/2=500)\n";

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 9. Empirical Benchmarking with `<chrono>`

### The Why
Sometimes you're not sure what an algorithm's complexity actually is — maybe it's someone else's code, maybe it's complex enough that the paper analysis is error-prone. Measuring it directly, across a few growing input sizes, gives you an empirical answer, and it's a skill worth having independent of theoretical analysis.

### Core Mechanism
- `<chrono>`'s `high_resolution_clock` gives you a timestamp before and after the work; `duration<double, milli>(end - start).count()` converts the difference to milliseconds.
- **The compiler will optimize away work it can prove is unused** — if you call a function and never use its result, an optimizing compiler (like with `-O2`) may delete the call ENTIRELY, and you'll measure "0 milliseconds" for real work. Declaring the result `volatile` (or otherwise using it, e.g. printing it) forces the compiler to actually keep the computation.
- **The empirical test for complexity class:** run the same algorithm at a few increasing input sizes and look at the RATIO of time increases, not the absolute numbers (which are noisy and hardware-dependent). Roughly doubling time when `n` doubles points to O(n); roughly quadrupling points to O(n²); barely increasing at all points to O(log n).
- Real measurements are noisy (other processes, cache effects, JIT/branch-prediction warmup) — don't expect the ratios to be exact. Look for the PATTERN across a few data points, not a single precise number.

### Common Pitfalls
- Benchmarking with the result unused and an optimizing compiler flag on, then concluding an algorithm is "instant" when really the compiler just deleted it — always sanity-check suspiciously fast benchmark results.
- Drawing conclusions from a single run — timing noise is real; run multiple trials or at least sanity-check the pattern across several input sizes before trusting the conclusion.


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
#include <chrono>
using namespace std;
using namespace std::chrono;

long long linearWork(int n) {
    long long sum = 0;
    for (int i = 0; i < n; i++) sum += i;
    return sum;
}

long long quadraticWork(int n) {
    long long sum = 0;
    for (int i = 0; i < n; i++)
        for (int j = 0; j < n; j++)
            sum += i + j;
    return sum;
}

template<typename Func>
double timeIt(Func f, int n) {
    auto start = high_resolution_clock::now();
    volatile long long result = f(n);  // volatile stops the compiler from optimizing the whole call away
    (void)result;
    auto end = high_resolution_clock::now();
    return duration<double, milli>(end - start).count();
}

int main() {
    // empirically confirm the theory: doubling n should roughly DOUBLE linear time,
    // but roughly QUADRUPLE quadratic time. Real measurements have noise, but the
    // pattern should be clearly visible across a few doublings.
    cout << "n\tlinear(ms)\tquadratic(ms)\n";
    for (int n : {2000, 4000, 8000}) {
        double linTime = timeIt(linearWork, n);
        double quadTime = timeIt(quadraticWork, n);
        cout << n << "\t" << linTime << "\t\t" << quadTime << "\n";
    }
    // this is the practical skill: when you're not sure of an algorithm's real complexity,
    // measure it across a few growing input sizes and look at the RATIO of time increases --
    // that ratio tells you the growth class empirically, without needing to prove it on paper

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 10. Phase Checkpoint, Cheat Sheet, and Answer Key

### Complexity Cheat Sheet

| Notation | Name | Example | n=10 | n=20 | n=40 |
|---|---|---|---|---|---|
| O(1) | Constant | Array access | 1 | 1 | 1 |
| O(log n) | Logarithmic | Binary search | ~3 | ~4 | ~5 |
| O(n) | Linear | Linear search | 10 | 20 | 40 |
| O(n log n) | Linearithmic | Efficient sort | ~33 | ~86 | ~213 |
| O(n²) | Quadratic | Nested loop, bubble sort | 100 | 400 | 1,600 |
| O(2ⁿ) | Exponential | Naive Fibonacci | 1,024 | ~1M | ~10¹² |
| O(n!) | Factorial | All permutations | 3.6M | 2.4×10¹⁸ | astronomically large |

### Checkpoint Questions
1. Why does Big-O ignore constant factors — why is `O(2n)` just written as `O(n)`?
2. A loop runs `for (int i = 1; i < n; i *= 3)`. What's its complexity, and why isn't it O(n)?
3. Two functions both have O(n) time complexity, but one is iterative and one is recursive. What's the one property they might NOT share, and why?
4. Using the aggregate method, explain in your own words why `vector::push_back` is amortized O(1) rather than exactly O(1) every call.
5. You benchmark an algorithm with `<chrono>` and get suspiciously fast, near-zero times even for large inputs. What's the first thing you should suspect?
6. When someone says "quicksort is O(n log n)," which case are they most likely referring to, and what's the actual worst case?

### Answer Key
1. Big-O describes growth CLASS for large n, not exact runtime. `2n` and `n` grow at the same rate as n increases — doubling n doubles both `2n` and `n` — so they belong to the same class. The constant `2` only shifts the line, it doesn't change its shape.
2. It's O(log₃ n) — still logarithmic, just with base 3 instead of base 2. The loop variable is multiplied each iteration rather than incremented by a fixed amount, so the number of iterations needed to reach `n` grows logarithmically, not linearly. (In Big-O, the base of the log doesn't even matter — O(log₃ n) and O(log₂ n) are the same complexity class, since logs of different bases differ only by a constant factor.)
3. Space complexity. Both can be O(n) in time, but the recursive version likely has O(n) auxiliary space from its call stack, while the iterative version can often be done in O(1) auxiliary space.
4. Resizes happen at sizes 1, 2, 4, 8, ... and each resize at size k copies k elements. Summed across all resizes up to n, that's a geometric series totaling just under n copy operations — not n² or anything worse. Adding the n individual push operations themselves, total work for n pushes is O(n), which averages to O(1) per push. Any single resizing push is genuinely expensive at that moment, but they're rare enough that the average stays constant.
5. That the compiler optimized away the "unused" work entirely, especially under `-O2` or higher. Check whether the computed result is actually used/read anywhere (or marked `volatile`) — if not, the compiler may have deleted the whole computation as dead code.
6. Average case. Quicksort's worst case is O(n²), which happens with a poor pivot choice on already-sorted (or adversarially-crafted) input — but randomized or median-of-three pivot selection makes that worst case exceedingly unlikely to hit in practice, which is why "O(n log n)" is the number people quote casually.

### Mini Project
Take any algorithm you've written before this course (or write a simple one, like a nested-loop duplicate checker) and:
1. State its time and space complexity from reading the code, before running anything.
2. Write a `<chrono>` benchmark across at least 3 growing input sizes to test your prediction empirically.
3. If the measured growth pattern doesn't match your prediction, figure out why — did you miscount a loop, or is the compiler optimizing something away?

This exercises: static complexity reasoning, the loop-analysis rules, and empirical verification with `<chrono>` — the full loop this notebook is built around.
